# x9_relevo — Topografia e Altimetria Municipal (Copernicus DEM 90m)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** elevação média, mínima e máxima (Copernicus GLO-90), m, municipal, estática  
**Conceito e Mecanismo Econômico:** Elevação média, mínima e máxima municipal (metros). Relevos planos facilitam a mecanização em larga escala da soja.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\topography__copdem90__municipio_candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar diferenças físicas; produto de superfície (DSM), com efeitos de vegetação/edificações.
> - Declividade e rugosidade exigem derivação dos rasters; não estão prontas nesta tabela.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`municipio`**: Nome do município.
- **`elevation_mean_m`**: Altitude média do município (metros).
- **`elevation_min_m`**: Altitude mínima.
- **`elevation_max_m`**: Altitude máxima.
- **`n_pixels`**: Número de pixels processados.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\topography__copdem90__municipio_candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x9_relevo`:
- **Atenção:** Explorar diferenças físicas; produto de superfície (DSM), com efeitos de vegetação/edificações.
- **Atenção:** Declividade e rugosidade exigem derivação dos rasters; não estão prontas nesta tabela.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
# O arquivo candidato original possui 142 municípios (inclui Boa Esperança do Norte - 5101837).
# Reincorporamos 5101837 em Nova Ubiratã (5106240) e recalculamos as estatísticas de altitude.
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO TXT: 'Explorar diferenças físicas; produto de superfície (DSM), com efeitos de vegetação/edificações.'
df = df.groupby('codigo_ibge', as_index=False).agg({
    'elevation_mean_m': 'mean',
    'elevation_min_m': 'min',
    'elevation_max_m': 'max'
})
print(f"Total de municípios de MT após harmonização: {df['codigo_ibge'].nunique()} (Meta: exatamente 141)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da altitude municipal em MT (metros):")
print(df[['elevation_mean_m', 'elevation_min_m', 'elevation_max_m']].describe())

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['elevation_mean_m'], bins=25, kde=True, ax=axes[0], color='peru')
axes[0].set_title("Distribuição da Altitude Média (m)")

df['amplitude_m'] = df['elevation_max_m'] - df['elevation_min_m']
sns.histplot(df['amplitude_m'], bins=25, kde=True, ax=axes[1], color='darkgoldenrod')
axes[1].set_title("Amplitude Altimétrica (Max - Min, metros)")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
sns.scatterplot(data=df, x='elevation_mean_m', y='amplitude_m', color='sienna')
plt.title("Altitude Média vs Amplitude Altimétrica nos Municípios de MT")
plt.xlabel("Altitude Média (m)")
plt.ylabel("Amplitude Altimétrica (m)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Fase 6: Feature Engineering — Topografia e Aptidão para Mecanização (SRTM)

### 1. Step 1: Características Topográficas Municipais
- A motomecanização moderna (plantio direto, colheitadeiras pesadas de grande porte) exige terrenos planos a suave-ondulados (declividade inferior a 8%).
- Terrenos acidentados ou com grande amplitude altimétrica inviabilizam a lavoura tecnificada de grãos, restringindo a terra à pecuária extensiva ou preservação.

### 2. Step 2: Interação Relevo x Preços (Shift-Share)
- **Amplitude Altimétrica:** $\Delta Altura_i = ElevacaoMax_i - ElevacaoMin_i$.
- **Dummy de Relevo Plano:** Vale 1 se a amplitude altimétrica for inferior a 250 metros.
- **Interação com Choque de Preços:** $\text{RelevoPlano}_i \times \Delta \ln(PrecoSoja_{t-1})$.
  Permite testar se a expansão da soja sobre a vegetação nativa avança preferencialmente sobre as chapadas planas mecanizáveis.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Indicadores Topográficos Municipais
df_relevo = df.copy()

# Amplitude altimétrica municipal (diferença entre ponto mais alto e mais baixo)
df_relevo['amplitude_altimetrica_m'] = df_relevo['elevation_max_m'] - df_relevo['elevation_min_m']

# dummy_relevo_plano: 1 se o relevo municipal for predominantemente suave/plano (amplitude < 250m)
df_relevo['dummy_relevo_plano'] = (df_relevo['amplitude_altimetrica_m'] < 250.0).astype(int)

# 2. Esqueleto de Interação Shift-Share para o Painel:
# ChoqueTopografia_it = dummy_relevo_plano_i * dln_psoja_lag1_t
df_relevo['exemplo_interacao_topografia_preco'] = df_relevo['dummy_relevo_plano'] * 0.15

print("Topografia e Aptidão à Motomecanização por Município:")
print(f"Municípios com relevo plano favorável à mecanização pesada: {df_relevo['dummy_relevo_plano'].sum()} / {len(df_relevo)}")
print(df_relevo[['codigo_ibge', 'elevation_mean_m', 'amplitude_altimetrica_m', 'dummy_relevo_plano']].head())